<a href="https://colab.research.google.com/github/dannys0n/Generative-AI-Works-CS394/blob/main/generate_synthetic_for_tts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generate Synthetic CS2 Commentary Training Data

Use this notebook to generate synthetic training rows for a Counter-Strike 2 commentary model.

The notebook expects seed examples shaped like the wrapper input expected:

```json
{
  "input": {
    "match_context": {...},
    "previous_events": [...],
    "current_events": [...],
    "overrides": {
      "caster": null,
      "prompt_style": null
    }
  }
}
```

It generates synthetic rows shaped like:

```json
{
  "input": {...},
  "output": {
    "commentary": "...",
    "prompt_style": "play_by_play_event",
    "caster": "play_by_play"
  }
}
```

This notebook is specifically for **CS2**. The only caster labels used are `play_by_play` and `color`.


## Data generation settings


In [ ]:
NUM_TRAIN_EXAMPLES = 0  # @param {type:"number"}
NUM_VAL_EXAMPLES = 0  # @param {type:"number"}
NUM_TEST_EXAMPLES = 5  # @param {type:"number"}
TEMPERATURE = 0.7  # @param {type:"number"}

DATA_FOLDER = "./.data/generated_cs2"
SEED_EXAMPLES_FILE = "./training_wrapper_pretty.jsonl"  # upload or mount this in Colab
MAX_SEED_EXAMPLES = 50  # @param {type:"number"}
SEED_EXAMPLES_PER_PROMPT = 3  # @param {type:"number"}

DATAGEN_URL = "https://openrouter.ai/api/v1"
DATAGEN_MODEL = "openai/gpt-5.4-nano"

ALLOWED_CASTERS = ["play_by_play", "color"]
ALLOWED_PROMPT_STYLES = [
    "play_by_play_event",
    "play_by_play_follow_up",
    "idle_color",
]

DEFAULT_OVERRIDES = {
    "caster": None,
    "prompt_style": None,
}

INLINE_SEED_EXAMPLES = []

from pathlib import Path
Path(DATA_FOLDER).mkdir(parents=True, exist_ok=True)


## Seed examples and schema


In [ ]:
import json
from pathlib import Path

def load_seed_examples(seed_file: str, inline_examples: list[dict], max_examples: int) -> list[dict]:
    examples: list[dict] = []
    path = Path(seed_file)

    if path.exists():
        text = path.read_text(encoding="utf-8")

        chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]
        if chunks:
            for chunk in chunks:
                record = json.loads(chunk)
                if isinstance(record, dict) and isinstance(record.get("input"), dict):
                    examples.append(record)
        else:
            for raw_line in text.splitlines():
                line = raw_line.strip()
                if not line:
                    continue
                record = json.loads(line)
                if isinstance(record, dict) and isinstance(record.get("input"), dict):
                    examples.append(record)

    for record in inline_examples:
        if isinstance(record, dict) and isinstance(record.get("input"), dict):
            examples.append(record)

    if max_examples > 0:
        examples = examples[:max_examples]

    return examples

SEED_EXAMPLES = load_seed_examples(SEED_EXAMPLES_FILE, INLINE_SEED_EXAMPLES, MAX_SEED_EXAMPLES)
print(f"Loaded {len(SEED_EXAMPLES)} seed examples")
if SEED_EXAMPLES:
    print(json.dumps(SEED_EXAMPLES[0], indent=2, sort_keys=True))

Loaded 50 seed examples
{
  "input": {
    "current_events": [
      {
        "event_type": "kill",
        "killer": {
          "kda": {
            "assists": 1,
            "deaths": 10,
            "kills": 10
          },
          "map_callout": "Blue",
          "name": "Azul",
          "round_kills": 2,
          "team": "CT"
        },
        "victim": {
          "name": "Panama",
          "team": "T"
        }
      },
      {
        "alive_counts_after": {
          "T": 4
        },
        "event_type": "team_counter"
      }
    ],
    "match_context": {
      "map_name": "de_dust2",
      "map_phase": "warmup",
      "round_number": 0,
      "round_phase": null,
      "score": {
        "CT": 0,
        "T": 0
      },
      "win_team": null
    },
    "overrides": {
      "caster": null,
      "prompt_style": null
    },
    "previous_events": []
  }
}


## Model for structured output


In [ ]:
from typing import Any, Literal
from pydantic import BaseModel, Field


class Overrides(BaseModel):
    caster: Literal["play_by_play", "color"] | None = None
    prompt_style: Literal["play_by_play_event", "play_by_play_follow_up", "idle_color"] | None = None


class SyntheticInput(BaseModel):
    match_context: dict[str, Any]
    previous_events: list[dict[str, Any]] = Field(default_factory=list)
    current_events: list[dict[str, Any]] = Field(default_factory=list)
    overrides: Overrides


class SyntheticOutput(BaseModel):
    commentary: str
    prompt_style: Literal["play_by_play_event", "play_by_play_follow_up", "idle_color"]
    caster: Literal["play_by_play", "color"]


class SyntheticTrainingRow(BaseModel):
    input: SyntheticInput
    output: SyntheticOutput


## Get OpenRouter API key


In [ ]:
import sys
import os
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
else:
  load_dotenv()

## Synthetic generation functions


## Dataset generation functions


In [ ]:
from tqdm import tqdm


def generate_dataset(num_examples: int, filename: str) -> None:
    if "SEED_EXAMPLES" not in globals():
        raise RuntimeError("SEED_EXAMPLES is not defined. Run the seed-loading cell first.")
    if not SEED_EXAMPLES:
        raise RuntimeError("SEED_EXAMPLES is empty. Check SEED_EXAMPLES_FILE and reload seeds.")

    with open(filename, "w", encoding="utf-8") as f:
        for idx in tqdm(range(num_examples)):
            row = None
            while row is None:
                try:
                    row = generate_synthetic_row(SEED_EXAMPLES)
                except Exception as error:
                    print(f"Error generating row {idx}: {error}")
                    raise
            f.write(row.model_dump_json() + "\n")
            f.flush()



## Few shot examples

In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        "input": {
            "match_context": {"map_name": "de_dust2", "round_number": 7, "score": {"CT": 2, "T": 5}},
            "previous_events": [],
            "current_events": [
                {
                    "event_type": "kill",
                    "killer": {"name": "Colin", "team": "CT", "map_callout": "B Doors"},
                    "victim": {"name": "Maru", "team": "T", "map_callout": "B Tunnels"}
                },
                {
                    "event_type": "team_counter",
                    "alive_counts_after": {"CT": 5, "T": 4}
                }
            ],
            "overrides": {"caster": "play_by_play", "prompt_style": "play_by_play_event"}
        },
        "output": {"commentary": "Colin opens at B Doors.", "prompt_style": "play_by_play_event", "caster": "play_by_play"}
    },
    {
        "input": {
            "match_context": {"map_name": "de_dust2", "round_number": 7, "score": {"CT": 2, "T": 5}},
            "previous_events": [
                {
                    "event_type": "kill",
                    "killer": {"name": "Colin", "team": "CT", "map_callout": "B Doors"},
                    "victim": {"name": "Maru", "team": "T", "map_callout": "B Tunnels"}
                },
                {
                    "event_type": "team_counter",
                    "alive_counts_after": {"CT": 5, "T": 4}
                }
            ],
            "current_events": [
                {
                    "event_type": "team_counter",
                    "alive_counts_after": {"CT": 5, "T": 3}
                }
            ],
            "overrides": {"caster": "play_by_play", "prompt_style": "play_by_play_follow_up"}
        },
        "output": {"commentary": "CTs keep the edge at B.", "prompt_style": "play_by_play_follow_up", "caster": "play_by_play"}
    },
    {
        "input": {
            "match_context": {"map_name": "de_dust2", "round_number": 10, "score": {"CT": 4, "T": 5}},
            "previous_events": [
                {
                    "event_type": "grenade_detonated",
                    "grenade_type": "smoke",
                    "detonation_callout": "Top Mid",
                    "owner_player": {"name": "Niles", "team": "T", "map_callout": "Mid"}
                }
            ],
            "current_events": [
                {
                    "event_type": "kill",
                    "killer": {"name": "Rook", "team": "T", "map_callout": "Catwalk"},
                    "victim": {"name": "Vera", "team": "CT", "map_callout": "Short"}
                }
            ],
            "overrides": {"caster": "play_by_play", "prompt_style": "play_by_play_event"}
        },
        "output": {"commentary": "Rook catches Vera on Short.", "prompt_style": "play_by_play_event", "caster": "play_by_play"}
    },
    {
        "input": {
            "match_context": {"map_name": "de_dust2", "round_number": 10, "score": {"CT": 4, "T": 5}},
            "previous_events": [
                {
                    "event_type": "grenade_detonated",
                    "grenade_type": "smoke",
                    "detonation_callout": "Top Mid",
                    "owner_player": {"name": "Niles", "team": "T", "map_callout": "Mid"}
                },
                {
                    "event_type": "kill",
                    "killer": {"name": "Rook", "team": "T", "map_callout": "Catwalk"},
                    "victim": {"name": "Vera", "team": "CT", "map_callout": "Short"}
                }
            ],
            "current_events": [
                {
                    "event_type": "bomb_event",
                    "state_after": "planted",
                    "site": "A"
                }
            ],
            "overrides": {"caster": "play_by_play", "prompt_style": "play_by_play_follow_up"}
        },
        "output": {"commentary": "Now the plant sticks at A.", "prompt_style": "play_by_play_follow_up", "caster": "play_by_play"}
    },
    {
        "input": {
            "match_context": {"map_name": "de_dust2", "round_number": 11, "score": {"CT": 6, "T": 5}},
            "previous_events": [
                {
                    "event_type": "bomb_event",
                    "state_after": "planted",
                    "site": "B"
                },
                {
                    "event_type": "team_counter",
                    "alive_counts_after": {"CT": 3, "T": 2}
                }
            ],
            "current_events": [],
            "overrides": {"caster": "color", "prompt_style": "idle_color"}
        },
        "output": {"commentary": "Three on two with no room left.", "prompt_style": "idle_color", "caster": "color"}
    },
    {
        "input": {
            "match_context": {"map_name": "de_dust2", "round_number": 12, "score": {"CT": 6, "T": 5}},
            "previous_events": [
                {
                    "event_type": "grenade_detonated",
                    "grenade_type": "smoke",
                    "detonation_callout": "Xbox",
                    "owner_player": {"name": "Niles", "team": "T", "map_callout": "Top Mid"}
                },
                {
                    "event_type": "grenade_detonated",
                    "grenade_type": "flash",
                    "detonation_callout": "Short",
                    "owner_player": {"name": "Mika", "team": "T", "map_callout": "Catwalk"}
                }
            ],
            "current_events": [],
            "overrides": {"caster": "color", "prompt_style": "idle_color"}
        },
        "output": {"commentary": "Mid utility is pulling CT eyes upward.", "prompt_style": "idle_color", "caster": "color"}
    }
]

## Generate all the data!


In [ ]:
import json
import openai
import os
import random

client = openai.OpenAI(
    base_url=DATAGEN_URL,
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

def build_seed_block(seed_examples: list[dict]) -> str:
    if not seed_examples:
        return "[]"
    return json.dumps(seed_examples, indent=2, sort_keys=True)


def build_few_shot_block() -> str:
    return json.dumps(FEW_SHOT_EXAMPLES, indent=2, sort_keys=True)


def choose_overrides() -> dict:
    caster = random.choice(ALLOWED_CASTERS + [None])
    prompt_style = random.choice(ALLOWED_PROMPT_STYLES + [None])

    if caster == "color" and prompt_style is None:
        prompt_style = "idle_color"
    if prompt_style == "idle_color" and caster is None:
        caster = "color"
    if prompt_style in {"play_by_play_event", "play_by_play_follow_up"} and caster is None:
        caster = "play_by_play"

    return {
        "caster": caster,
        "prompt_style": prompt_style,
    }


def extract_json_object(raw_text: str) -> dict:
    raw_text = raw_text.strip()
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        start = raw_text.find("{")
        end = raw_text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise ValueError("Model did not return a valid JSON object")
        return json.loads(raw_text[start:end+1])


def generate_synthetic_row(seed_examples: list[dict]) -> SyntheticTrainingRow:
    if not seed_examples:
        raise ValueError("No seed examples loaded. Upload or point SEED_EXAMPLES_FILE to v3 training wrapper JSONL.")

    sampled_examples = random.sample(seed_examples, k=min(SEED_EXAMPLES_PER_PROMPT, len(seed_examples)))
    overrides = choose_overrides()

    prompt = f'''
You are generating synthetic supervised fine-tuning data for a Counter-Strike 2 commentary model.

Important context:
- This is CS2, not generic esports.
- The only caster labels are: play_by_play, color.
- Do not mention Portal, Turret, Announcer, or any character personalities.
- Keep commentary grounded in current_events.
- previous_events is light context only and must never override current_events.
- The generated row must stay plausible for Counter-Strike 2.
- output.commentary must be exactly one short sentence for TTS.

Length rules:
- play_by_play_event: ideally 2 to 6 words, never exceed 8 words.
- play_by_play_follow_up: ideally 3 to 8 words, never exceed 10 words.
- idle_color: ideally 4 to 10 words, never exceed 12 words.

Style guidance:
- play_by_play is direct, clipped, factual.
- color is dry, observant, understated.
- Prefer exact player names and callouts when present.
- Prefer literal, speakable phrasing over dramatic phrasing.
- Avoid generic filler like: locks it down, makes a play, finds a frag, huge kill, big round, what a play, comes up big.

Few-shot examples:
{build_few_shot_block()}

Seed examples for structure and event diversity:
{build_seed_block(sampled_examples)}

Generate exactly one new synthetic row with this JSON structure:
{{
  "input": {{
    "match_context": {{...}},
    "previous_events": [...],
    "current_events": [...],
    "overrides": {{
      "caster": "play_by_play" | "color" | null,
      "prompt_style": "play_by_play_event" | "play_by_play_follow_up" | "idle_color" | null
    }}
  }},
  "output": {{
    "commentary": "short natural commentary line",
    "prompt_style": "play_by_play_event" | "play_by_play_follow_up" | "idle_color",
    "caster": "play_by_play" | "color"
  }}
}}

Hard requirements:
- input.overrides must equal this exact object: {json.dumps(overrides, sort_keys=True)}
- output.prompt_style and output.caster must respect input.overrides whenever they are not null.
- output.commentary must match the selected prompt style.
- Return JSON only.
'''

    response = client.responses.create(
        model=DATAGEN_MODEL,
        input=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
    )

    parsed = extract_json_object(response.output_text)
    return SyntheticTrainingRow.model_validate(parsed)


In [ ]:
from datetime import datetime

TRAIN_FILE = f"{DATA_FOLDER}/train_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
VALID_FILE = f"{DATA_FOLDER}/valid_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
TEST_FILE = f"{DATA_FOLDER}/test_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"

generate_dataset(NUM_TRAIN_EXAMPLES, TRAIN_FILE)
print(TRAIN_FILE)

generate_dataset(NUM_VAL_EXAMPLES, VALID_FILE)
print(VALID_FILE)

generate_dataset(NUM_TEST_EXAMPLES, TEST_FILE)
print(TEST_FILE)



0it [00:00, ?it/s]


./.data/generated_cs2/train_2026-04-12-16:32:06.jsonl


0it [00:00, ?it/s]


./.data/generated_cs2/valid_2026-04-12-16:32:06.jsonl


100%|██████████| 5/5 [00:12<00:00,  2.53s/it]

./.data/generated_cs2/test_2026-04-12-16:32:06.jsonl


In [ ]:
# Preview generated rows
import json
from pathlib import Path

preview_path = Path(TEST_FILE)
if preview_path.exists():
    for raw_line in preview_path.read_text(encoding='utf-8').splitlines()[:3]:
        print(json.dumps(json.loads(raw_line), indent=2, sort_keys=True))
        print('---')


{
  "input": {
    "current_events": [
      {
        "event_type": "kill",
        "killer": {
          "map_callout": "B Doors",
          "name": "Maru",
          "team": "T"
        },
        "victim": {
          "map_callout": "B Tunnels",
          "name": "Azul",
          "team": "CT"
        }
      }
    ],
    "match_context": {
      "map_name": "de_dust2",
      "map_phase": "live",
      "round_number": 8,
      "round_phase": "live",
      "score": {
        "CT": 3,
        "T": 5
      },
      "win_team": null
    },
    "overrides": {
      "caster": "color",
      "prompt_style": "play_by_play_event"
    },
    "previous_events": [
      {
        "event_type": "kill",
        "killer": {
          "map_callout": "Mid",
          "name": "Conleth",
          "team": "CT"
        },
        "victim": {
          "map_callout": "Short",
          "name": "Walt",
          "team": "T"
        }
      }
    ]
  },
  "output": {
    "caster": "color",
    "commentar